# Step 0: Raw Data Exploration

This notebook documents the PhysioNet GAITNDD v1.0.0 dataset in its raw, unprocessed state. Its purpose is to establish ground truth about what the raw files contain before any filtering or feature engineering is applied.

**Dataset:** Gait in Neurodegenerative Disease Database (Hausdorff et al., 2000). 64 subjects across four groups: Parkinson's Disease (PD), Huntington's Disease (HD), ALS, and healthy controls. Each subject has one `.ts` file containing one stride per row.

**File format:** Plain-text, tab-separated, no header, 13 columns.
- Column 1: elapsed recording time in seconds (not a feature, discarded during loading)
- Columns 2-13: the 12 gait timing features (left/right stride, swing, stance intervals and percentages, double support interval and percentage)

**What this notebook covers:**
- Subject file counts per condition
- Raw file structure (column count, separator, representative rows)
- Total raw stride count
- Pause event distribution (stride interval > 3.0 s)
- hunt20 sensor failure (right-foot sensor frozen, entire subject unusable)
- als5/als7 partial sensor malfunction (double_support_pct > 100)

No filtering or feature engineering is performed here. See `01_preprocessing.ipynb` for the processed version.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import polars as pl
from preprocessing import FEATURE_COLS

RAW_DIR = Path('../data/raw/gait-in-neurodegenerative-disease-database-1.0.0')

# Column names for raw loading (13 columns: elapsed_s + 12 features).
RAW_COLS = ['elapsed_s'] + FEATURE_COLS

In [2]:
# Count .ts files per condition by filename prefix.
prefix_map = {'park': 'PD', 'hunt': 'HD', 'als': 'ALS', 'control': 'Control'}
counts = {label: 0 for label in prefix_map.values()}

for f in RAW_DIR.glob('*.ts'):
    for prefix, label in prefix_map.items():
        if f.stem.startswith(prefix):
            counts[label] += 1
            break

print('Subject file counts per condition:')
for label, n in counts.items():
    print(f'  {label}: {n}')
print(f'  Total: {sum(counts.values())}')

Subject file counts per condition:
  PD: 15
  HD: 20
  ALS: 13
  Control: 16
  Total: 64


In [3]:
# Load park1.ts as a representative file and inspect its raw structure.
park1 = pl.read_csv(
    RAW_DIR / 'park1.ts',
    separator='\t',
    has_header=False,
    new_columns=RAW_COLS,
    schema_overrides={col: pl.Float64 for col in RAW_COLS},
)

print(f'park1.ts: {park1.shape[0]} rows, {park1.shape[1]} columns')
print(f'Column names: {park1.columns}')
print()
print('First 3 rows (including elapsed_s in column 1):')
print(park1.head(3))

park1.ts: 245 rows, 13 columns
Column names: ['elapsed_s', 'left_stride_s', 'right_stride_s', 'left_swing_s', 'right_swing_s', 'left_swing_pct', 'right_swing_pct', 'left_stance_s', 'right_stance_s', 'left_stance_pct', 'right_stance_pct', 'double_support_s', 'double_support_pct']

First 3 rows (including elapsed_s in column 1):
shape: (3, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ elapsed_s ┆ left_stri ┆ right_str ┆ left_swin ┆ … ┆ left_stan ┆ right_sta ┆ double_su ┆ double_s │
│ ---       ┆ de_s      ┆ ide_s     ┆ g_s       ┆   ┆ ce_pct    ┆ nce_pct   ┆ pport_s   ┆ upport_p │
│ f64       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ct       │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═

In [4]:
# Total raw stride count across all 64 files.
total_rows = 0
per_file_counts = []

for ts_file in sorted(RAW_DIR.glob('*.ts')):
    n = sum(1 for _ in open(ts_file))
    total_rows += n
    per_file_counts.append((ts_file.stem, n))

print(f'Total raw strides across all files: {total_rows}')
print()
print('Strides per subject (sorted by subject name):')
for subject_id, n in per_file_counts:
    print(f'  {subject_id}: {n}')

Total raw strides across all files: 15160

Strides per subject (sorted by subject name):
  als1: 194
  als10: 246
  als11: 229
  als12: 122
  als13: 183
  als2: 242
  als3: 215
  als4: 135
  als5: 205
  als6: 176
  als7: 159
  als8: 232
  als9: 212
  control1: 259
  control10: 277
  control11: 269
  control12: 244
  control13: 251
  control14: 249
  control15: 198
  control16: 250
  control2: 241
  control3: 255
  control4: 267
  control5: 250
  control6: 270
  control7: 260
  control8: 261
  control9: 275
  hunt1: 310
  hunt10: 220
  hunt11: 239
  hunt12: 258
  hunt13: 167
  hunt14: 255
  hunt15: 217
  hunt16: 190
  hunt17: 248
  hunt18: 252
  hunt19: 243
  hunt2: 225
  hunt20: 238
  hunt3: 232
  hunt4: 268
  hunt5: 263
  hunt6: 263
  hunt7: 232
  hunt8: 256
  hunt9: 270
  park1: 245
  park10: 288
  park11: 230
  park12: 247
  park13: 251
  park14: 278
  park15: 237
  park2: 277
  park3: 230
  park4: 222
  park5: 263
  park6: 269
  park7: 226
  park8: 203
  park9: 222


In [5]:
# Pause event distribution: rows where left_stride_s > 3.0 OR right_stride_s > 3.0.
# These represent recording intervals where the subject stopped walking.
pause_counts = []

for ts_file in sorted(RAW_DIR.glob('*.ts')):
    df = pl.read_csv(
        ts_file,
        separator='\t',
        has_header=False,
        new_columns=RAW_COLS,
        schema_overrides={col: pl.Float64 for col in RAW_COLS},
    )
    n_pause = df.filter(
        (pl.col('left_stride_s') > 3.0) | (pl.col('right_stride_s') > 3.0)
    ).height
    if n_pause > 0:
        pause_counts.append((ts_file.stem, n_pause))

total_pause = sum(n for _, n in pause_counts)
print(f'Total rows removed by stride filter (> 3.0 s): {total_pause}')
print()
print('Per-file breakdown (files with at least one pause-event row):')
for subject_id, n in sorted(pause_counts, key=lambda x: -x[1]):
    print(f'  {subject_id}: {n} rows')

Total rows removed by stride filter (> 3.0 s): 276

Per-file breakdown (files with at least one pause-event row):
  hunt20: 238 rows
  park11: 8 rows
  als3: 6 rows
  als4: 6 rows
  als12: 5 rows
  hunt18: 3 rows
  hunt13: 2 rows
  hunt16: 2 rows
  park7: 2 rows
  als1: 1 rows
  als5: 1 rows
  hunt14: 1 rows
  hunt15: 1 rows


In [6]:
# hunt20 sensor failure: the right-foot sensor recorded frozen values
# between 19.6 s and 58.4 s for the entire file, making all 238 rows unusable.
# The left-foot sensor recorded normal stride values throughout.
hunt20 = pl.read_csv(
    RAW_DIR / 'hunt20.ts',
    separator='\t',
    has_header=False,
    new_columns=RAW_COLS,
    schema_overrides={col: pl.Float64 for col in RAW_COLS},
)

right_vals = hunt20['right_stride_s']
left_vals  = hunt20['left_stride_s']

print(f'hunt20: {hunt20.height} total rows')
print()
print(f'left_stride_s  min={left_vals.min():.4f}  max={left_vals.max():.4f}  (normal range)')
print(f'right_stride_s min={right_vals.min():.4f}  max={right_vals.max():.4f}  (frozen sensor)')
print()
print(f'Unique right_stride_s values (frozen to a small set of large constants):')
print(hunt20['right_stride_s'].unique().sort())
print()
print(f'Rows caught by stride filter (right_stride_s > 3.0): {hunt20.filter(pl.col("right_stride_s") > 3.0).height}')
print('All 238 rows are removed. hunt20 contributes 0 usable strides after filtering.')

hunt20: 238 total rows

left_stride_s  min=0.9133  max=1.3067  (normal range)
right_stride_s min=19.6433  max=58.3867  (frozen sensor)

Unique right_stride_s values (frozen to a small set of large constants):
shape: (6,)
Series: 'right_stride_s' [f64]
[
	19.6433
	40.09
	42.91
	57.09
	57.5567
	58.3867
]

Rows caught by stride filter (right_stride_s > 3.0): 238
All 238 rows are removed. hunt20 contributes 0 usable strides after filtering.


In [7]:
# als5 and als7 partial sensor malfunction: double_support_pct > 100.
# These rows pass the stride interval filter (both stride values are < 3.0 s)
# but have physically impossible percentage values caused by a partially frozen
# right-foot sensor producing corrupted derived timing values.
#
# als5: right_stride_s frozen at 1.2533 s mid-recording; right-side derived
#       percentages become corrupted, pushing double_support_pct above 100.
# als7: double_support_s exceeds the stride interval, producing pct > 100.

subjects_of_interest = ['als5', 'als7']

for subject_id in subjects_of_interest:
    df = pl.read_csv(
        RAW_DIR / f'{subject_id}.ts',
        separator='\t',
        has_header=False,
        new_columns=RAW_COLS,
        schema_overrides={col: pl.Float64 for col in RAW_COLS},
    )
    bad_rows = df.filter(pl.col('double_support_pct') > 100.0)
    passes_stride_filter = bad_rows.filter(
        (pl.col('left_stride_s') <= 3.0) & (pl.col('right_stride_s') <= 3.0)
    )

    print(f'{subject_id}: {df.height} total rows')
    print(f'  Rows with double_support_pct > 100:    {bad_rows.height}')
    print(f'  Of those, passing stride filter (<= 3.0 s): {passes_stride_filter.height}')
    print(f'  double_support_pct range in bad rows: '
          f'min={bad_rows["double_support_pct"].min():.1f}  '
          f'max={bad_rows["double_support_pct"].max():.1f}')
    print(f'  Sample row:')
    print(f'    left_stride_s={bad_rows["left_stride_s"][0]:.4f}  '
          f'right_stride_s={bad_rows["right_stride_s"][0]:.4f}  '
          f'double_support_pct={bad_rows["double_support_pct"][0]:.2f}')
    print()

als5: 205 total rows
  Rows with double_support_pct > 100:    91
  Of those, passing stride filter (<= 3.0 s): 90
  double_support_pct range in bad rows: min=100.7  max=134.3
  Sample row:
    left_stride_s=1.5300  right_stride_s=1.2533  double_support_pct=100.65

als7: 159 total rows
  Rows with double_support_pct > 100:    36
  Of those, passing stride filter (<= 3.0 s): 36
  double_support_pct range in bad rows: min=128.4  max=161.4
  Sample row:
    left_stride_s=1.7900  right_stride_s=1.7600  double_support_pct=158.47



## Raw Data Findings Summary

### Subject counts
64 subjects total: PD=15, HD=20, ALS=13, Control=16. All expected files are present.

### File structure
Every `.ts` file is tab-separated, 13 columns, no header. Column 1 is elapsed recording time (discarded). Columns 2-13 are the 12 gait timing features in a consistent order across all subjects and conditions.

### Total raw strides
15,160 rows across all 64 files before any filtering.

### Artifact type 1: pause events (stride interval > 3.0 s)
276 rows across 13 files have at least one stride interval exceeding 3.0 seconds. These are recording intervals where the subject stopped walking; the equipment continued logging, producing entries with intervals of 5-58 seconds that are not gait cycles.

The dominant contributor is hunt20 (238 of the 276 rows), which is not a pause event but a complete right-foot sensor failure: every row in the file has a frozen right_stride_s value in the range 19.6-58.4 s. The stride interval filter removes the entire file, leaving 19 usable HD subjects.

### Artifact type 2: physically impossible percentage values (double_support_pct > 100)
131 rows pass the stride interval filter but have double_support_pct > 100, which is physically impossible (a percentage of a stride cycle cannot exceed 100). These arise from partial sensor malfunction where the right-foot sensor partially froze mid-recording, causing corrupted derived timing values. The main contributors are als5 (90 rows) and als7 (36 rows).

### The two filters in preprocessing
These two artifact classes motivate the two-stage filter in `filter_pause_events()`:
1. Remove rows where left_stride_s > 3.0 OR right_stride_s > 3.0 (catches hunt20 and genuine pauses).
2. Remove rows where double_support_pct > 100 (catches als5/als7 partial malfunction rows that the stride threshold misses).

Combined: 407 rows removed (2.7% of 15,160), leaving 14,753 clean strides.